Assignment for Day 2

1. Load ES futures data (from folder market_data: ES[M][yy].csv)
2. Apply futures roll method to adjust ESZ25
3. Load SPY data from yf (fallback: SPX.csv)
4. Visualize histogram of daily returns
5. Calculate skewness and excess kurtosis

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import yfinance as yf

TICKERS = ['ESM25', 'ESU25', 'ESZ25', 'ESH26', 'ESM26', 'ESU26']


In [2]:
data = {}
close_prices = []
volumes = []
# load futures data in folder market_data
for ticker in TICKERS:
    print(f"Loading data for {ticker}")
    try:
        data[ticker] = pd.read_csv(f"../../market_data/futures/{ticker}.csv")
        data[ticker].set_index("Time", inplace=True)
        data[ticker].index = pd.to_datetime(data[ticker].index)
        # extract close prices and concatenate
        dfc = data[ticker][['Latest']]
        dfc.columns = [ticker]
        close_prices.append(dfc)
        # extract volumes and concatenate
        dfv = data[ticker][['Volume']]
        dfv.columns = [ticker]
        volumes.append(dfv)
    except FileNotFoundError:
        print(f"File not found for {ticker}")

Loading data for ESM25
Loading data for ESU25
Loading data for ESZ25
Loading data for ESH26
Loading data for ESM26
Loading data for ESU26


In [3]:
data[TICKERS[0]].head()

,Open,High,Low,Latest,Change,%Change,Volume,Open Int
Time,,,,,,,,
2025-06-20,5984.00,6019.25,5918.25,6010.21,28.71,0.48%,71380,0
2025-06-18,5977.25,6020.75,5964.75,5981.50,-3.50,-0.06%,453191,562716
2025-06-17,6038.50,6040.50,5976.75,5985.00,-50.75,-0.84%,870571,667682
2025-06-16,5949.00,6055.25,5944.00,6035.75,56.50,0.94%,1744867,1017789
2025-06-13,6045.00,6045.00,5927.50,5979.25,-70.25,-1.16%,2162167,1664937


In [4]:
df_close = pd.concat(close_prices, axis=1)
df_close.tail()


/var/folders/82/hh8lkwfx2s5bvc9lnq2rp_0w0000gn/T/ipykernel_32434/3764384865.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_close = pd.concat(close_prices, axis=1)


,ESM25,ESU25,ESZ25,ESH26,ESM26,ESU26
Time,,,,,,
2026-02-25,NaN,NaN,NaN,6959.75,7013.25,7062.50
2026-02-26,NaN,NaN,NaN,6920.00,6972.75,7023.75
2026-02-27,NaN,NaN,NaN,6889.00,6941.50,6994.00
2026-03-02,NaN,NaN,NaN,6888.25,6940.75,6991.50
2026-03-03,NaN,NaN,NaN,6755.00,6811.00,6875.50


In [5]:
# plot unadjusted close prices using plotly
fig = px.line(df_close, y=df_close.columns, title='Unadjusted Close Prices')
fig.update_layout(xaxis_title=None, yaxis_title="Price")
fig.update_layout(legend_title='Contract')
fig.show()


In [6]:
# plot volumes as bar charts
df_volumes = pd.concat(volumes, axis=1)

fig = px.bar(df_volumes, y=df_volumes.columns, title='Volumes by ES Futures Contract')
fig.update_layout(xaxis_title=None, yaxis_title="Volume")
fig.update_layout(legend_title_text='Contract')
fig.show()



/var/folders/82/hh8lkwfx2s5bvc9lnq2rp_0w0000gn/T/ipykernel_32434/3847511861.py:2: Pandas4Warning:

Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.



In [7]:
# adjust the prices of ESZ25 by rolling to ESH26 on 12-11-2025 (at Close)
# load ESH26 data
df = df_close[['ESZ25', 'ESH26']]
df.head()

# create a new column for the adjusted prices
roll_date = pd.to_datetime('2025-12-11')
roll_gap = df.loc[roll_date, 'ESH26'] - df.loc[roll_date, 'ESZ25']
df['ESZ25_adjusted'] = df['ESZ25'] + roll_gap
df.loc[roll_date:, 'ESZ25_adjusted'] = np.nan

# plot the adjusted prices
fig = px.line(df[df.index>= "2025-09-22"], y=['ESZ25', 'ESZ25_adjusted', 'ESH26'], title='Adjusted Prices')
fig.update_layout(xaxis_title=None, yaxis_title="Price")
fig.update_layout(legend_title='Contract')
fig.show()


In [8]:
# load SPY prices
spy = yf.download("SPY", start="2023-01-01", end="2026-03-13")
# drop 'Ticker' level from columns
spy = spy.droplevel(1, axis=1)
spy.head()

/opt/miniconda3/envs/algotrading/lib/python3.11/site-packages/yfinance/scrapers/history.py:172: Pandas4Warning:

Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.



YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed

1 Failed download:
['SPY']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,


In [ ]:
# fallback if yf fails: use SPX.csv as proxy for SPY
if spy.empty:
    spy = pd.read_csv('../../market_data/daily/SPX.csv')
    # spy.set_index("Date", inplace=True)    # inplace=True should be avoided
    spy = spy.set_index("Date")

In [ ]:
spy.index = pd.to_datetime(spy.index)
spy = spy.sort_index(ascending=True)
spy.head()

# calculate daily log returns
spy['Log Return'] = np.log(spy['Close'] / spy['Close'].shift(1))    # shift direction depends on sorting

In [18]:
spy.describe()

,Open,High,Low,Close,Change,Volume,Log Return
count,519.000000,519.000000,519.000000,519.000000,519.000000,519.0,518.000000
mean,5952.210886,5981.201137,5920.357861,5953.165183,3.610443,0.0,0.000616
std,591.115745,589.552578,591.426163,590.626506,56.635033,0.0,0.009969
min,4950.160000,4957.770000,4835.040000,4953.170000,-322.440000,0.0,-0.061609
25%,5490.120000,5506.610000,5449.465000,5480.385000,-19.345000,0.0,-0.003308
50%,5910.180000,5938.370000,5866.070000,5912.170000,5.860000,0.0,0.001027
75%,6454.075000,6475.650000,6439.565000,6463.100000,33.285000,0.0,0.005544
max,7002.000000,7002.240000,6963.460000,6978.600000,474.130000,0.0,0.090895


In [19]:
# plot SPX Latest prices
fig = px.line(spy, y='Close', title='SPX Index')
fig.show()

# plot SPX Log Returns
fig = px.line(spy, y='Log Return', title='SPX Log Returns')
fig.show()

In [20]:
# fit lognormal distribution to log returns
log_returns = spy['Log Return']
mean = log_returns.mean()
std = log_returns.std()

# plot histogram of log returns and fitted lognormal distribution
fig = px.histogram(spy, x='Log Return', title='SPX Log Returns Histogram', nbins=100)
fig.show()

# plot the pdf of the lognormal distribution
x = np.linspace(log_returns.min(), log_returns.max(), 100)
pdf = 1/(std*np.sqrt(2*np.pi))*np.exp(-(x-mean)**2/(2*std**2))
px.line(x=x, y=pdf)

In [21]:
# calculate skewness of log returns as moment of order 3
log_returns = spy['Log Return']
mean = log_returns.mean()
std = log_returns.std()
skewness = ((log_returns - mean)**3).mean() / std**3
print(f"Mean of SPX Log Returns: {mean}")
print(f"Skewness of SPX Log Returns: {skewness}")

# calculate excess kurtosis of log returns as moment of order 4 minus 3 (actual formula)
kurtosis = ((log_returns - mean)**4).mean() / std**4
excess_kurtosis = kurtosis - 3
print(f"Excess Kurtosis of SPX Log Returns: {excess_kurtosis}")

Mean of SPX Log Returns: 0.0006160684511854698
Skewness of SPX Log Returns: 0.42767570363945023
Excess Kurtosis of SPX Log Returns: 16.647774647891957


In [22]:
# Load SPY daily datar = spy["Close"].pct_change().dropna()          # daily log-approx returns

# ── Moments ─────────────────────────────────────────────────────────
mu    = log_returns.mean()                                # daily mean
sigma = log_returns.std()                                 # daily std dev
skew  = log_returns.skew()                                # skewness
kurt  = log_returns.kurtosis()                            # excess kurtosis

print(f"Mean (daily):     {mu:.4%}")
print(f"Std deviation:       {sigma:.4%}  (annualized: {sigma*np.sqrt(252):.2%})")
print(f"Skewness:         {skew:.3f}   (normal = 0)")
print(f"Excess Kurtosis:  {kurt:.3f}   (normal kurtosis = 3, excess = 0)")


Mean (daily):     0.0616%
Std deviation:       0.9969%  (annualized: 15.83%)
Skewness:         0.430   (normal = 0)
Excess Kurtosis:  16.898   (normal kurtosis = 3, excess = 0)
